In [ ]:
from typing import Union
from faster_whisper import WhisperModel, BatchedInferencePipeline
import numpy as np
import pysbd
import kss
import librosa

In [ ]:
MODEL_SIZE = "large-v3"

SAMPLE_RATE = 16000
BUFFER_SIZE = 10

In [ ]:
class KSSTokenizer:
  # KSSTokenizer likes Segmenter
  def segment(self, text):
    # Use kss to segment the text
    return kss.split_sentences(text)

In [ ]:
class Tokenizer:
  tokenizers:dict[pysbd.Segmenter] = {}
    
  @staticmethod
  def get_tokenizer(lang: str) -> pysbd.Segmenter:
    # whisper와 지원하는 언어 차이를 확인해야 한다.
    lang = lang.strip()
    if lang not in Tokenizer.tokenizers:
      if lang == "ko":
        Tokenizer.tokenizers[lang] = KSSTokenizer()
      else:
        try:
          Tokenizer.tokenizers[lang] = pysbd.Segmenter(language=lang)
        except Exception as e:
          print(f"Tokenizer not found for {lang}. Error: {e}. 이거 whisper하고 비교해서 제대로 되게 만들어야 함")
          return None
    return Tokenizer.tokenizers[lang]

In [ ]:
class Whisper:
  def __init__(self, beam_size: int = 5):
    self.__BEAM_SIZE = beam_size
    self.__model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="int8")

  def translate(self, audio: np.ndarray, language:str = None, prompt: str = ""):
    return self.__model.transcribe(
      audio, beam_size=self.__BEAM_SIZE, language=language,
      word_timestamps=True, vad_filter=True,
      initial_prompt=prompt
    )

In [ ]:
whisper = Whisper()

In [ ]:
audio, sr = librosa.load(".data/news_with_english.mp3", sr=SAMPLE_RATE)

In [ ]:
total_samples = len(audio)

segments = []
pos = 0
while pos < total_samples:
  rand_len = int(np.random.normal(loc=16000, scale=400))
  rand_len = np.clip(rand_len, 14000, 18000)
  end = min(pos + rand_len, total_samples)

  chunk = audio[pos:end]
  segments.append(chunk)
  pos = end

In [ ]:
# full_text = ""
# for segment in segments:
#   if len(segment) < 160:
#     continue
#   seg, info = whisper.translate(segment, language="ko")
#   for s in seg:
#     full_text += s.text

In [ ]:
# print(full_text)

In [ ]:
from schemas.whisper import Segment

class Sentence:
  def __init__(self, lang:str, text:str, audio:np.ndarray, tokens:list[Segment]):
    self.__lang = lang
    self.__text = text
    self.__audio = audio
    self.__tokens = tokens
  
  def __str__(self):
    return f"{self.__lang} {self.__text} {self.__tokens}"

  @property
  def lang(self):
    return self.__lang
  @property
  def text(self):
    return self.__text
  @property
  def audio(self):
    return self.__audio
  @property
  def tokens(self):
    return self.__tokens

In [ ]:
class InferenceContext:
  def __init__(
    self,
    max_pre_recog:int = 2,
    sample_rate:int = SAMPLE_RATE,
  ):
    self.__MAX_PRE_RECOG = max_pre_recog
    self.__SAMPLE_RATE = sample_rate

  def get_prompt(self, pre_sentence:Sentence = None):
    return pre_sentence.text if pre_sentence else ""
  
  def get_audio(self, audio:np.ndarray, pre_recog:list[Sentence] = []):
    new_audio = np.zeros(0, dtype=np.float32)
    for sentence in pre_recog:
      new_audio = np.concatenate([new_audio, sentence.audio])
    new_audio = np.concatenate([new_audio, audio])
    return new_audio

  def get_new(self, audio, segments, info):
    lan = info.language
    tokenizer = Tokenizer.get_tokenizer(lan)

    sentences = []
    end = 0
    for idx, segment in enumerate(segments, 1):
      text = segment.text
      words = segment.words
      start = end
      end = segment.end if len(segments) > idx else len(audio) / self.__SAMPLE_RATE

      if tokenizer is None:
        print(f"Tokenizer not found for {lan}.")
        sentences.append(Sentence(
          lan, 
          text,
          audio[int(start * self.__SAMPLE_RATE):int(end * self.__SAMPLE_RATE)],
          words
        ))
        continue

      scents = tokenizer.segment(text)
      idx_end = 0
      ed = start
      for scent in scents:
        idx_start = idx_end
        idx_end = idx_start + len(scent.split(" "))

        st = ed
        ed = words[idx_end].start if len(words) > idx_end else end

        ad_st = int(st * self.__SAMPLE_RATE)
        ad_ed = int(ed * self.__SAMPLE_RATE)
        ad = audio[ad_st:ad_ed]
        sentences.append(Sentence(
          lan, 
          scent, 
          ad,
          words[idx_start:idx_end]
        ))
      
      end = ed

    return (
      sentences[:-self.__MAX_PRE_RECOG], 
      sentences[-(self.__MAX_PRE_RECOG + 1)] if len(sentences) > self.__MAX_PRE_RECOG else None,
      sentences[-self.__MAX_PRE_RECOG:]
    )
      

In [ ]:
class AudioRecognition:

  def __init__(self, inference_context:InferenceContext, model:Whisper):
    self.inference_context = inference_context
    self.model = model
  
  def recognition(
    self,
    audio:np.ndarray,
    order:int,
    pre_sentence:Sentence = None,
    pre_recog: list[Sentence] = [],
    language:str = None
  ):
    prompt = self.inference_context.get_prompt(pre_sentence)
    audio = self.inference_context.get_audio(audio, pre_recog)

    seg, info = self.model.translate(audio, language)
    completed, pre_sentence_, pre_recog = self.inference_context.get_new(audio, list(seg), info)
    
    completed_dict = {}
    for o, sentence in enumerate(completed, order):
      completed_dict[o] = sentence

    if pre_sentence_:
      pre_sentence = pre_sentence_
    
    return completed_dict, order + len(completed), pre_sentence, pre_recog

In [ ]:
inference_context = InferenceContext()
recognition = AudioRecognition(inference_context, whisper)

In [ ]:
raise Exception("stop")

In [ ]:
from IPython.display import Audio

In [ ]:
order = 0
pre_sentence = None
pre_recog = []
completed_sents = {}

In [ ]:
for segment in segments:
  completed_dict, order, pre_sentence, pre_recog = recognition.recognition(segment, order, pre_sentence, pre_recog)
  completed_sents.update(completed_dict)

  # print("-" * 20)
  # print("completed_sent", [(k, completed_sents[k].text) for k in completed_sents])
  # print("completed_dict", [(k, completed_dict[k].text) for k in completed_dict])
  # print("pre_sentence", pre_sentence.text if pre_sentence else "")
  # print("pre_recog", [k.text for k in pre_recog])

for sent in pre_recog:
  completed_sents[order] = sent
  order += 1

In [ ]:
Audio(segment, rate=SAMPLE_RATE)

In [ ]:
# audio = np.concatenate([s.audio for s in pre_recog])
Audio(audio, rate=SAMPLE_RATE)

In [ ]:
audio_ = np.concatenate([s.audio for s in pre_recog])
Audio(audio_, rate=SAMPLE_RATE)

In [ ]:
for key, item in completed_sents.items():
  print(key, item.text)